### 🔧 SVM with KNN Neighborhood  

###  Import Libraries and setup

In [ ]:
from sklearn.neighbors import NearestNeighbors
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
import joblib
import os
import warnings

warnings.filterwarnings('ignore')

In [ ]:
# Set random seeds for reproducibility
SEED = 42


### Load Preprocessed Data

In [ ]:
X_train = joblib.load('../models/X_train.pkl')
X_test = joblib.load('../models/X_test.pkl')
y_train = joblib.load('../models/y_train.pkl')
y_test = joblib.load('../models/y_test.pkl')

### Convert sparse matrices to dense arrays 

In [ ]:
X_train_dense = X_train.toarray() if hasattr(X_train, 'toarray') else X_train
X_test_dense = X_test.toarray() if hasattr(X_test, 'toarray') else X_test

### Fixed Hyperparameters 

In [ ]:
k = 11
C = 0.33142972574893137
kernel = 'linear'
gamma = 'auto'

### Fit KNN Model

In [ ]:
nbrs = NearestNeighbors(n_neighbors=k).fit(X_train_dense)

### Test Predictions

In [ ]:
fallback_label = y_train.value_counts().idxmax()
test_predictions = []
for test_point in X_test_dense:
    indices = nbrs.kneighbors([test_point], return_distance=False)[0]
    y_local = y_train.iloc[indices]
    if y_local.nunique() == 1:
        test_predictions.append(fallback_label)
        continue
    clf = SVC(C=C, kernel=kernel, gamma=gamma, random_state=SEED)
    clf.fit(X_train_dense[indices], y_local)
    test_predictions.append(clf.predict([test_point])[0])

### Evaluation Results

In [ ]:
print("\n📊 Test Accuracy:", accuracy_score(y_test, test_predictions))
print("\n📄 Test Classification Report:\n", classification_report(y_test, test_predictions))